<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E/22/384 — Bayesian Inference Assignment




# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

The platform assumes a two-parameter logistic (2PL) item response model:
$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}}$$

with prior $\Theta \sim \mathscr{N}(0,1)$, and the posterior at step $k-1$ becomes the prior at step $k$.

## Task 1 — Visualizing the Mechanics

We plot $p_i(\theta)$ against $\theta$ for two distinct discrimination values $a_i$. One of these ($a_i = 1.5$) is paired with three different difficulty values $b_i \in \{-2, 0, 2\}$, so we can see the joint effect of $a$ and $b$.

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

def p_2pl(theta, a, b):
    """2PL item response probability."""
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

theta_vals = np.linspace(-6, 6, 400)

curve_specs = [
    {"a": 0.4, "b": 0,  "label": "a=0.4, b=0",  "dash": "dot"},
    {"a": 1.5, "b": -2, "label": "a=1.5, b=-2", "dash": "solid"},
    {"a": 1.5, "b": 0,  "label": "a=1.5, b=0",  "dash": "solid"},
    {"a": 1.5, "b": 2,  "label": "a=1.5, b=2",  "dash": "solid"},
]

fig = go.Figure()
for spec in curve_specs:
    y = p_2pl(theta_vals, spec["a"], spec["b"])
    fig.add_trace(go.Scatter(x=theta_vals, y=y, mode="lines",
                              name=spec["label"], line=dict(dash=spec["dash"])))

fig.update_layout(
    title="2PL Item Response Curves: Effect of Discrimination (a) and Difficulty (b)",
    xaxis_title="Latent ability θ",
    yaxis_title="P(Y=1 | θ)",
    template="plotly_white",
    legend_title="Item parameters",
)
fig.show()

**Interpretation for Visualizing the Mechanics**
===
The item difficulty $b_i$ is the value of $\theta$ at which the response probability equals exactly $0.5$ — it is the horizontal location of the curve's inflection point. Increasing $b_i$ shifts the whole logistic curve to the right without changing its shape, meaning a more able user ($\theta$) is now needed to have even odds of answering correctly. The discrimination $a_i$ controls how steeply the curve rises around $\theta = b_i$: the flat, low-$a$ curve ($a=0.4$) barely distinguishes between weak and strong users, whereas the steep, high-$a$ curves ($a=1.5$) sharply separate users just below vs. just above the difficulty threshold.

## Task 2 — Sequential Likelihood Contribution

**Likelihood of a single new response.** Given $\Theta = \theta$, the response $y_k \in \{0,1\}$ to item $k$ is Bernoulli with success probability $p_k(\theta)$, so its likelihood contribution can be written compactly as

$$L(y_k \mid \theta) = p_k(\theta)^{y_k}\,\big[1-p_k(\theta)\big]^{1-y_k}, \qquad p_k(\theta) = \frac{1}{1+e^{-a_k(\theta - b_k)}}.$$

**Joint likelihood of the running history.** Assuming the responses are conditionally independent given $\Theta=\theta$ (a standard IRT assumption — each item's difficulty/discrimination is fixed and known, and the only source of randomness given $\theta$ is the item-specific Bernoulli draw), the joint likelihood of $\mathbf{y}^{(k)} = (y_1,\dots,y_k)$ is the product of the individual contributions:

$$L\big(\mathbf{y}^{(k)} \mid \theta\big) = \prod_{i=1}^{k} p_i(\theta)^{y_i}\big[1-p_i(\theta)\big]^{1-y_i}.$$

## Task 3 — Mathematical Formulation of the Running Update

By Bayes' theorem, the posterior after $k$ observations is proportional to the product of the likelihood of all $k$ responses and the initial prior:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \;\propto\; L\big(\mathbf{y}^{(k)}\mid\theta\big)\, f_{\Theta}^{(0)}(\theta).$$

Because $L(\mathbf{y}^{(k)}\mid\theta) = L(y_k\mid\theta)\cdot L(\mathbf{y}^{(k-1)}\mid\theta)$, and $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}) \propto L(\mathbf{y}^{(k-1)}\mid\theta) f_\Theta^{(0)}(\theta)$, substituting gives the **recursive** (sequential Bayesian) update:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \;\propto\; L(y_k \mid \theta)\; f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}).$$

That is, the previous posterior plays the role of the prior for the current step, and it only needs to be re-weighted by the likelihood contribution of the single new observation $y_k$ — the full response history does not need to be re-processed at every step.

## Task 4 — Dynamic Shifting

When $y_k = 1$ (a correct answer) on a highly difficult item (large $b_k$), the likelihood factor is $L(y_k=1\mid\theta) = p_k(\theta) = \frac{1}{1+e^{-a_k(\theta-b_k)}}$. This function is small for $\theta \ll b_k$ and only becomes appreciable once $\theta$ approaches or exceeds $b_k$. Multiplying the previous posterior by this weight therefore **down-weights the probability mass at low $\theta$ and up-weights the mass at $\theta \gtrsim b_k$**, since only sufficiently able users have a non-negligible chance of answering a hard item correctly.

The net effect is that the peak (and mean) of the running posterior shifts to the **right**, toward higher ability values, relative to the previous step — and the shift is larger the harder the item is (the larger $b_k$ is relative to the previous posterior mode).

## Task 5 — Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls the steepness of $p_k(\theta)$ around $\theta = b_k$, i.e. how informative the observation is.

- **Large $a_k$:** the likelihood $L(y_k\mid\theta)$ transitions almost like a step function around $b_k$ — it is close to $0$ on one side and close to $1$ on the other. Multiplying the prior by such a sharp function strongly suppresses probability mass on the "wrong" side of $b_k$, which **narrows (sharpens) the posterior**, i.e. reduces its variance substantially — a single highly discriminating item can be very informative.
- **Small $a_k$:** the likelihood is a nearly flat function of $\theta$ (close to a constant $\approx 0.5$ everywhere), so multiplying by it barely changes the shape of the prior. The posterior **remains close to the previous posterior**, and its variance shrinks only marginally — a low-discrimination item carries little information about $\theta$.

## Task 6 — Numerical Implementation of a Running Grid

Since the 2PL likelihood is not conjugate to the Gaussian prior, we maintain the posterior numerically on a fixed grid:

1. **Grid setup.** Choose a fine, evenly spaced grid $\theta_1, \dots, \theta_M$ spanning a plausible ability range (e.g. $[-5, 5]$), with spacing $\Delta\theta$.
2. **Initialize.** Evaluate the prior density on the grid: $f^{(0)}[m] = f_\Theta^{(0)}(\theta_m) = \phi(\theta_m)$ (standard normal pdf), for $m = 1,\dots,M$.
3. **Sequential update.** On observing item $k$ with parameters $(a_k, b_k)$ and response $y_k$:
   - Compute $p_k(\theta_m) = 1/(1+e^{-a_k(\theta_m-b_k)})$ on the grid.
   - Compute the unnormalized posterior $\tilde f^{(k)}[m] = p_k(\theta_m)^{y_k}\,[1-p_k(\theta_m)]^{1-y_k} \cdot f^{(k-1)}[m]$.
4. **Sequential normalization.** Numerically integrate the unnormalized posterior over the grid (e.g. via the trapezoidal rule, `np.trapezoid(f_tilde, theta_grid)`) to get the normalizing constant $Z_k$, then set $f^{(k)}[m] = \tilde f^{(k)}[m] / Z_k$ so that $\int f^{(k)}(\theta)\,d\theta \approx 1$.
5. **Repeat** steps 3–4 for each new item, always using the previous step's normalized posterior as the new prior.
6. **Point estimates** at any step are then read off the grid: posterior mean $\approx \sum_m \theta_m f^{(k)}[m]\Delta\theta$, and MAP $\approx \theta_{m^*}$ where $m^* = \arg\max_m f^{(k)}[m]$.

## Task 7 — Evaluating Convergence over the Timeline

We simulate $n=20$ items answered by a user with true hidden ability $\theta_{\text{true}} = 0.75$, tracking the running posterior mean and MAP estimate at every step.

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

rng = np.random.default_rng(384)  # seeded with registration number for reproducibility

theta_true = 0.75
n_items = 20

# Fine grid over plausible ability range
theta_grid = np.linspace(-6, 6, 1000)
d_theta = theta_grid[1] - theta_grid[0]

# Initialize prior: standard normal N(0,1)
posterior = stats.norm.pdf(theta_grid, loc=0, scale=1)
posterior /= np.trapezoid(posterior, theta_grid)  # ensure normalized

def p_2pl(theta, a, b):
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

# Storage for estimators, including step 0 (prior-only) values
bayes_means = [np.trapezoid(theta_grid * posterior, theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior)]]

items_a = []
items_b = []
items_y = []

for k in range(1, n_items + 1):
    # Random item parameters
    a_k = rng.uniform(0.5, 2.0)
    b_k = rng.normal(0, 1)

    # Simulate the true response probability and draw the response
    true_prob = p_2pl(theta_true, a_k, b_k)
    y_k = 1 if rng.uniform(0, 1) < true_prob else 0

    items_a.append(a_k); items_b.append(b_k); items_y.append(y_k)

    # Likelihood contribution of this single item on the grid
    p_grid = p_2pl(theta_grid, a_k, b_k)
    likelihood = p_grid**y_k * (1 - p_grid)**(1 - y_k)

    # Sequential Bayesian update
    unnormalized = likelihood * posterior
    Z_k = np.trapezoid(unnormalized, theta_grid)
    posterior = unnormalized / Z_k

    # Track running estimators
    bayes_means.append(np.trapezoid(theta_grid * posterior, theta_grid))
    map_estimates.append(theta_grid[np.argmax(posterior)])

steps = np.arange(0, n_items + 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=bayes_means, mode="lines+markers", name="Posterior Mean (Bayes estimate)"))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode="lines+markers", name="MAP estimate"))
fig.add_hline(y=theta_true, line_dash="dash", line_color="black",
              annotation_text="θ_true = 0.75", annotation_position="bottom right")

fig.update_layout(
    title=f"Running Ability Estimates over {n_items} Items (θ_true = {theta_true})",
    xaxis_title="Item step k",
    yaxis_title="Estimated ability",
    template="plotly_white",
)
fig.show()

print("Simulated responses (0=incorrect, 1=correct):", items_y)
print(f"Final Bayes estimate: {bayes_means[-1]:.4f}")
print(f"Final MAP estimate:   {map_estimates[-1]:.4f}")

Simulated responses (0=incorrect, 1=correct): [1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1]
Final Bayes estimate: 1.1088
Final MAP estimate:   1.0871


**Analysis.** At step $k=0$ both estimators start at $0$ (the prior mean/mode), so the initial distance from $\theta_{\text{true}}=0.75$ is largest. As more items are answered, each new response contributes an independent piece of evidence, and — since responses are on average generated according to the true ability — the running posterior mean and MAP estimate both fluctuate but trend toward $\theta_{\text{true}}$, with the gap shrinking (on average) as $k$ increases, particularly once several high-discrimination items have been observed. The two estimators also converge toward each other as $k$ grows, because with enough data the posterior becomes increasingly concentrated and approximately symmetric (by the Bayesian analogue of the CLT), making the mean and mode nearly coincide. This shrinking spread reflects growing confidence: the platform's uncertainty about the user's ability decreases monotonically (in expectation) as evidence accumulates, even though individual noisy responses can cause short-term wobble in the estimates.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform updates its belief about an advertisement's true click-through rate $\theta \in [0,1]$ one impression at a time, using a Beta prior $\Theta \sim \text{Beta}(\alpha_0,\beta_0)$ and Bernoulli observations $Y_k \mid \Theta=\theta \sim \text{Bernoulli}(\theta)$.

## Task 1 — Structural Probability and Properties

In [5]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

theta_grid_beta = np.linspace(0, 1, 500)

beta_configs = [
    {"alpha": 1, "beta": 1, "label": "Uninformative: Beta(1,1)"},
    {"alpha": 2, "beta": 8, "label": "Right-skewed: Beta(2,8)"},
    {"alpha": 8, "beta": 2, "label": "Left-skewed: Beta(8,2)"},
]

fig = go.Figure()
for cfg in beta_configs:
    pdf_vals = stats.beta.pdf(theta_grid_beta, cfg["alpha"], cfg["beta"])
    fig.add_trace(go.Scatter(x=theta_grid_beta, y=pdf_vals, mode="lines", name=cfg["label"]))

fig.update_layout(
    title="Beta(α, β) Densities for Different Parameter Pairs",
    xaxis_title="θ",
    yaxis_title="Density",
    template="plotly_white",
)
fig.show()

**Interpretation.** The mean of a $\text{Beta}(\alpha,\beta)$ distribution is $\alpha/(\alpha+\beta)$, so the balance between $\alpha$ and $\beta$ directly controls where the density's center of mass sits on $[0,1]$. $\text{Beta}(1,1)$ is the uniform distribution — flat, with no preference over $\theta$. $\text{Beta}(2,8)$ has $\alpha < \beta$, so its mass concentrates toward small $\theta$ (right-skewed tail, but center-of-mass shifted left) — appropriate for a belief that clicks are rare. $\text{Beta}(8,2)$ has $\alpha > \beta$, so its mass concentrates toward large $\theta$ — a belief that clicks are common. As $\alpha$ grows relative to $\beta$ the density shifts right and becomes more sharply peaked; as $\beta$ grows relative to $\alpha$ it shifts left.

## Task 2 — Sequential Likelihood and Joint History

**Single-response likelihood.** Since $Y_k \mid \Theta=\theta \sim \text{Bernoulli}(\theta)$,

$$L(y_k \mid \theta) = \theta^{y_k}(1-\theta)^{1-y_k}.$$

**Joint likelihood.** Assuming the impressions are conditionally i.i.d. given $\theta$,

$$L\big(\mathbf{y}^{(k)} \mid \theta\big) = \prod_{i=1}^{k}\theta^{y_i}(1-\theta)^{1-y_i} = \theta^{\sum_{i=1}^k y_i}(1-\theta)^{k - \sum_{i=1}^k y_i}.$$

That is, the joint likelihood depends on the data only through the total number of clicks $s_k=\sum_{i=1}^k y_i$ observed in the first $k$ impressions — this sufficient-statistic structure is exactly what makes conjugacy possible.

## Task 3 — Closed-Form Analytical Updates (Conjugacy)

By Bayes' theorem, using the previous posterior $\text{Beta}(\alpha_{k-1},\beta_{k-1})$ as the prior at step $k$:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) \;\propto\; L(y_k\mid\theta)\, f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)}) = \theta^{y_k}(1-\theta)^{1-y_k}\cdot \theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}.$$

Combining the exponents of $\theta$ and $(1-\theta)$:

$$f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)}) \;\propto\; \theta^{(\alpha_{k-1}+y_k)-1}\,(1-\theta)^{(\beta_{k-1}+1-y_k)-1},$$

which is (up to normalization) exactly the kernel of a $\text{Beta}(\alpha_k, \beta_k)$ density. This proves **Beta–Binomial conjugacy**: the posterior stays in the Beta family, with the simple arithmetic (closed-form) parameter updates

$$\alpha_k = \alpha_{k-1} + y_k, \qquad \beta_k = \beta_{k-1} + (1-y_k).$$

Equivalently, after $k$ observed impressions with $s_k$ total clicks: $\alpha_k = \alpha_0 + s_k$, $\beta_k = \beta_0 + (k-s_k)$.

**Posterior mean.** For $\Theta\mid\mathbf{Y}^{(k)} \sim \text{Beta}(\alpha_k,\beta_k)$,

$$\mathbb{E}\big[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}\big] = \frac{\alpha_k}{\alpha_k+\beta_k}.$$

## Task 4 — Dynamic Shifting Mechanics

Since $\alpha_k = \alpha_{k-1}+y_k$ and $\beta_k=\beta_{k-1}+(1-y_k)$:

- A **click** ($y_k=1$) increments only $\alpha$ (leaving $\beta$ unchanged), which increases the posterior mean $\alpha_k/(\alpha_k+\beta_k)$ and shifts the peak of the Beta density to the **right**.
- A **non-click** ($y_k=0$) increments only $\beta$, decreasing the posterior mean and shifting the peak to the **left**.

In both cases the total "concentration" $\alpha_k+\beta_k$ also grows by exactly $1$ per observation, so the density becomes progressively narrower (more confident) as impressions accumulate, regardless of which outcome occurs.

**Contrast with non-conjugate models.** In the 2PL IRT setting of the first question, the Bernoulli likelihood is combined with a Gaussian prior, and the resulting posterior has no simple named form — every update requires re-evaluating the likelihood and prior on a grid and renormalizing numerically (e.g. via the trapezoidal rule). Here, by contrast, because the Beta prior is conjugate to the Bernoulli/Binomial likelihood, the *entire* posterior is captured by just two numbers $(\alpha_k,\beta_k)$, updated with simple integer arithmetic — no grid, no numerical integration, and no approximation error at any step.

## Task 5 — Running Point Estimators

Directly from the updated shape parameters $\alpha_k,\beta_k$ of $\text{Beta}(\alpha_k,\beta_k)$:

**Running Posterior Mean (Bayes estimate):**
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k+\beta_k}$$

**Running MAP estimate** (mode of the Beta density, valid for $\alpha_k,\beta_k>1$; the density is otherwise monotonic/U-shaped and the mode lies at a boundary):
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k-1}{\alpha_k+\beta_k-2}, \qquad \alpha_k,\beta_k>1.$$

## Task 6 — Performance Tracking and Convergence Analysis

We simulate $n=100$ impressions from an advertisement with true hidden CTR $\theta_{\text{true}}=0.35$, starting from an uninformative $\text{Beta}(1,1)$ prior, and track the closed-form running estimators.

In [4]:
import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(384)  # seeded with registration number

theta_true_ctr = 0.35
n_impressions = 100

alpha_0, beta_0 = 1, 1
alpha_k, beta_k = alpha_0, beta_0

# Step-0 (prior-only) estimators
bayes_estimates = [alpha_k / (alpha_k + beta_k)]
map_estimates_ctr = [np.nan]  # Beta(1,1) is flat -> MAP undefined/non-unique at step 0

clicks = []
for k in range(1, n_impressions + 1):
    y_k = 1 if rng.uniform(0, 1) < theta_true_ctr else 0
    clicks.append(y_k)

    # Closed-form conjugate update
    alpha_k = alpha_k + y_k
    beta_k = beta_k + (1 - y_k)

    bayes_estimates.append(alpha_k / (alpha_k + beta_k))
    if alpha_k > 1 and beta_k > 1:
        map_estimates_ctr.append((alpha_k - 1) / (alpha_k + beta_k - 2))
    else:
        map_estimates_ctr.append(np.nan)

steps_ctr = np.arange(0, n_impressions + 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps_ctr, y=bayes_estimates, mode="lines", name="Posterior Mean (Bayes estimate)"))
fig.add_trace(go.Scatter(x=steps_ctr, y=map_estimates_ctr, mode="lines", name="MAP estimate"))
fig.add_hline(y=theta_true_ctr, line_dash="dash", line_color="black",
              annotation_text="θ_true = 0.35", annotation_position="bottom right")

fig.update_layout(
    title=f"Running CTR Estimates over {n_impressions} Impressions (θ_true = {theta_true_ctr})",
    xaxis_title="Impression step k",
    yaxis_title="Estimated click-through rate",
    template="plotly_white",
)
fig.show()

print(f"Total clicks observed: {sum(clicks)} out of {n_impressions}")
print(f"Final alpha_k, beta_k: {alpha_k}, {beta_k}")
print(f"Final Bayes estimate: {bayes_estimates[-1]:.4f}")
print(f"Final MAP estimate:   {map_estimates_ctr[-1]:.4f}")

Total clicks observed: 36 out of 100
Final alpha_k, beta_k: 37, 65
Final Bayes estimate: 0.3627
Final MAP estimate:   0.3600


**Analysis.** Starting from the uninformative $\text{Beta}(1,1)$ prior (mean $0.5$, far from $\theta_{\text{true}}=0.35$), the earliest few impressions cause large, jumpy swings in the estimate because each single observation still carries a lot of relative weight against a total concentration $\alpha_k+\beta_k$ that is still small. As $k$ increases toward $100$, both $\alpha_k$ and $\beta_k$ grow roughly in proportion to $k$, so each additional observation contributes a progressively smaller fractional update — the estimates settle down and the distance $|\widehat{\theta}^{(k)}-\theta_{\text{true}}|$ shrinks on average, following a rate on the order of $1/\sqrt{k}$ (since the posterior variance of a Beta distribution shrinks like $O(1/k)$). The Bayes mean and MAP estimate also converge toward each other, since with enough data the Beta posterior becomes increasingly symmetric and concentrated around its mean. This illustrates the general Bayesian principle that the influence of the initial prior *washes out* as data accumulates: no matter what reasonable $(\alpha_0,\beta_0)$ we start from, given enough impressions the likelihood dominates and the posterior concentrates tightly around the true CTR.